# Day 10 · 阶段复习与购物车项目

> **前置**: Day01-09 全部(print/input/变量/字符串/分支/循环/函数/列表字典/文件 I/O/异常/OOP/模块/生成器/装饰器)
> **关键问题**: 前 9 天学了"语法点",今天把它们**焊接成一个完整控制台应用**;同时通过复习把暴露出的高频错误集中歼灭

**引入(5 分钟)**

**第 1 讲:三大高频错题回顾**

**错题 1 —— 条件分支边界错误**:`elif score >= 80` 永远执行不到,因为 `>= 60` 先吃掉了所有 >= 60 的分数。正确做法是从高到低判断:先 `>= 80`,再 `>= 60`。

**错题 2 —— 累加器位置错误**:`total = 0` 写在循环里,每次迭代都被清零,结果只加了最后一个。正确做法是累加器在循环**外**初始化。

**错题 3 —— return vs print**:函数 `print` 了结果,`return` 隐式返回 `None`,其他函数拿不到结果。正确做法是用 `return` 把值交出去。

排查方法论:**报错不慌张 -> 看最后一行异常类型 -> 看报错行号 -> `print(type(变量))` + `print(变量)`**。

In [ ]:
# ===== 三个错题的正确写法示范 =====

# 正确写法 1:条件分支从高到低判断
score = 85
if score >= 80:
    print("良好")               # 正确执行到这里
elif score >= 60:
    print("及格")

# 正确写法 2:累加器在循环外初始化
total = 0                       # 循环外初始化,只执行一次
for price in [5.5, 12.8, 8.0]:
    total += price              # 循环内只累加,不清零
print(f"总价: {total}")         # 总价: 26.3

# 正确写法 3:return 把值交出去
def calc(prices):
    return sum(prices)          # return 而不是 print

result = calc([5.5, 12.8, 8.0])
print(f"结果: {result}")        # 结果: 26.3(其他函数能拿到)

---

**第 2 讲:购物车系统 v1 —— 函数封装 + 异常加固 + JSON 持久化**

需求 checklist:商品列表(字典 + 列表) / 菜单循环 / 同商品数量累加 / 结算 + 总价 / 异常加固 / JSON 持久化。口诀:**框架先把函数名填上,每个函数只做一件事,main 只做调度**。

In [ ]:
import json, os

# ===== 商品库与购物车为全局变量 =====
商品库 = [
    {"id": 1, "name": "苹果", "price": 5.50},
    {"id": 2, "name": "牛奶", "price": 12.80},
    {"id": 3, "name": "面包", "price": 8.00},
]
购物车 = []       # 每项: {"id", "name", "price", "qty"}
FILE = "cart.json"

# ===== JSON 持久化函数 =====
def load_cart():
    # 启动时加载购物车,文件不存在或内容非法返回空列表
    if not os.path.exists(FILE):
        return []
    with open(FILE, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []

def save_cart():
    # 保存购物车到 JSON 文件
    with open(FILE, "w", encoding="utf-8") as f:
        json.dump(购物车, f, ensure_ascii=False, indent=2)

# ===== 核心功能函数 =====
def 查找商品(编号):
    # 根据编号在商品库中查找,找不到返回 None
    for 商品 in 商品库:
        if 商品["id"] == 编号:
            return 商品
    return None

def 浏览商品():
    # 打印所有商品信息
    for 商品 in 商品库:
        print(f"{商品['id']}. {商品['name']} ￥{商品['price']}")

def 加入购物车():
    # 输入商品编号,已存在则数量 +1,不存在则新增条目
    try:
        编号 = int(input("请输入商品编号:"))
    except ValueError:
        print("请输入数字!")        # 非法输入友好提示,不崩溃
        return
    商品 = 查找商品(编号)
    if not 商品:
        print("商品不存在!")
        return
    # 购物车中已有该商品 -> 数量 +1
    for 项 in 购物车:
        if 项["id"] == 编号:
            项["qty"] += 1
            save_cart()
            return
    # 购物车中没有 -> 新增条目
    购物车.append({"id": 编号, "name": 商品["name"],
                   "price": 商品["price"], "qty": 1})
    save_cart()

def 结算():
    # 打印每条商品的小计和总价,然后清空购物车
    if not 购物车:
        print("购物车为空!")
        return
    总价 = 0
    for 项 in 购物车:
        小计 = 项["price"] * 项["qty"]
        print(f"{项['name']} x{项['qty']} = ￥{小计:.2f}")
        总价 += 小计
    print(f"总价: ￥{总价:.2f}")
    购物车.clear()              # 清空购物车(原地修改,不是赋值 [])
    save_cart()

# 启动时加载购物车
购物车 = load_cart()

---

**第 3 讲:主循环 main() 调度**

口诀:**框架先把函数名填上,每个函数只做一件事,main 只做调度**。`input` 永远返回字符串,和数字比较必须 `int()` 转;函数内改列表用 `.clear()` / `append()` 等原地方法,不要赋值 `lst = []`(那会创建新局部变量,原列表没变)。

In [ ]:
def main():
    # 主循环:显示菜单,根据输入调度对应功能
    while True:
        print("\n1.浏览商品 2.加入购物车 3.结算 0.退出")
        选 = input("请选择:")
        if 选 == "0":
            print("再见!")
            break
        elif 选 == "1":
            浏览商品()
        elif 选 == "2":
            加入购物车()
        elif 选 == "3":
            结算()
        else:
            print("无效选项!")

if __name__ == "__main__":
    main()

---

**今日小结**

把 Day01-09 语法焊接成完整项目。三大高频错:条件边界 / 累加器位置 / return vs print。

易错点:`input` 永远返回字符串,比较必须 `int()` 转;函数内改列表用 `.clear()` / `append()` 等原地方法,不要赋值 `lst = []`;循环内初始化累加器,每次迭代都被清零;全局变量在函数内只读不需要 `global`,要改才需要声明 `global`。

**更多练习**

- 当堂练:course/lesson10/in_class/practice01-06.py
- 课后作业:course/lesson10/homework/task01-03.py